# Phi-3-mini Financial Fine-tuning (QLoRA via Unsloth)

**Runtime:** Google Colab T4 (free tier)  
**Expected duration:** 30-45 min for 500 samples, 3 epochs  
**Output:** LoRA adapter pushed to HuggingFace Hub private repo

## Pipeline position
```
ragdb → export CronJob → JSONL → [this notebook] → HF Hub adapter → vLLM on-cluster → LiteLLM phi3-financial-ft
```

## Before running
1. Export training data from the cluster:  
   `kubectl create job ft-export-manual --from=cronjob/finetuning-data-export -n ai`  
   `kubectl cp ai/<pod>:/export/training-YYYYMMDD.jsonl ./training-data.jsonl`
2. Upload `training-data.jsonl` to this Colab session (Files panel, left sidebar)
3. Set your HuggingFace token and repo name in the Config cell below

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install unsloth mlflow huggingface_hub --quiet
!pip install --upgrade --no-cache-dir unsloth --quiet

In [ ]:
# ── 2. Config — edit these ───────────────────────────────────────────────────
HF_TOKEN       = "hf_xxxx"                           # HuggingFace write token
HF_REPO        = "andrelair-platform/phi3-financial-ft"  # private repo target
MLFLOW_URI     = "https://mlflow.devandre.sbs"        # your on-cluster MLflow
TRAINING_FILE  = "training-data.jsonl"                # uploaded file path
MODEL_NAME     = "microsoft/Phi-3-mini-4k-instruct"

# QLoRA hyperparams
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
MAX_SEQ_LEN    = 1024
BATCH_SIZE     = 4
GRAD_ACCUM     = 4      # effective batch = 16
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3
WARMUP_STEPS   = 10

In [ ]:
# ── 3. Load model with Unsloth 4-bit QLoRA ───────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto (bfloat16 on Ampere, float16 on T4)
    load_in_4bit=True,   # QLoRA: 4-bit base, train adapter in float16
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ── 4. Load + format training data ───────────────────────────────────────────
import json
from datasets import Dataset

# Alpaca prompt template — matches what phi3-financial uses in production
ALPACA_TEMPLATE = """Below is an instruction from a financial professional, with optional context.
Write a clear, accurate response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS = tokenizer.eos_token

def format_record(row):
    text = ALPACA_TEMPLATE.format(
        instruction=row["instruction"],
        input=row.get("input", ""),
        output=row["output"],
    ) + EOS
    return {"text": text}

records = []
with open(TRAINING_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

dataset = Dataset.from_list(records).map(format_record)
print(f"Training examples: {len(dataset)}")
print("Sample:")
print(dataset[0]["text"][:300])

In [ ]:
# ── 5. Train with MLflow tracking ────────────────────────────────────────────
import mlflow
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("phi3-financial-finetuning")

training_args = TrainingArguments(
    output_dir="/tmp/phi3-ft-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",   # we log manually below
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

with mlflow.start_run(run_name=f"phi3-financial-ft-r{LORA_RANK}") as run:
    mlflow.log_params({
        "base_model": MODEL_NAME,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRAD_ACCUM,
        "train_samples": len(dataset),
        "hf_repo": HF_REPO,
    })

    trainer_stats = trainer.train()

    mlflow.log_metrics({
        "train_loss": trainer_stats.training_loss,
        "train_runtime_min": trainer_stats.metrics["train_runtime"] / 60,
        "samples_per_second": trainer_stats.metrics["train_samples_per_second"],
    })

    RUN_ID = run.info.run_id
    print(f"MLflow run ID: {RUN_ID}")
    print(f"Train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── 6. Push adapter to HuggingFace Hub ───────────────────────────────────────
from huggingface_hub import login

login(token=HF_TOKEN)

# Tag adapter with MLflow run ID for full traceability
commit_message = f"QLoRA adapter — mlflow run {RUN_ID} — loss {trainer_stats.training_loss:.4f}"

model.save_pretrained_merged(
    "/tmp/phi3-ft-adapter",
    tokenizer,
    save_method="lora",   # adapter-only, not merged weights (~50MB)
)

model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method="lora",
    token=HF_TOKEN,
    commit_message=commit_message,
    private=True,
)

# Log HF repo link in MLflow for the registry entry
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("hf_adapter_commit", commit_message)
    mlflow.set_tag("adapter_ready", "true")

print(f"Adapter pushed to: https://huggingface.co/{HF_REPO}")
print()
print("Next steps (on-cluster):")
print(f"  1. Update 38-vllm.yaml: add --lora-modules phi3-financial-ft={HF_REPO}")
print(f"  2. Open PR → ArgoCD syncs → vLLM hot-reloads adapter")
print(f"  3. Test: curl http://vllm.ai.svc.cluster.local:8000/v1/completions ...")

In [ ]:
# ── 7. Quick inference test before pushing ───────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_TEMPLATE.format(
    instruction="What is the CET1 ratio requirement under Basel III?",
    input="",
    output="",
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)[len(test_prompt):])